# Подход 1. Строгий датасет связанных объектов

Этот notebook содержит полный текст SQL 41 и запускает его напрямую. Внешний SQL-файл для работы notebook не нужен.

В результат попадают только объекты недвижимости, для которых подтверждена цепочка:

`договор → заявка → выбранная задача → связь с объектом → характеристики → условия`.

Преимущество подхода — у каждой строки есть договорный контекст. Недостаток — большая часть договоров не имеет заполненной связи с объектами и не попадает в датасет.

## 1. Библиотеки

Следующую ячейку достаточно выполнить один раз в используемом окружении. Если библиотеки уже установлены, её можно пропустить.

In [ ]:
%pip install pandas sqlalchemy "psycopg[binary]" oracledb

In [ ]:
import getpass
from pathlib import Path
import oracledb
import pandas as pd
from sqlalchemy import URL, create_engine, text

pd.set_option('display.max_columns', 100)

## 2. Путь к проекту

Ячейка сама найдёт корень проекта, если notebook открыт из папки проекта или `notebooks`.

In [ ]:
def find_project_root(start):
    start = Path(start).resolve()
    for folder in (start, *start.parents):
        if (folder / 'docs' / 'PROJECT_CONTEXT.md').is_file():
            return folder
    raise FileNotFoundError('Не найден корень проекта')

PROJECT_ROOT = find_project_root(Path.cwd())
OUTPUT_DIR = PROJECT_ROOT / 'РЕЗУЛЬТАТЫ_ЛОКАЛЬНО'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Корень проекта:', PROJECT_ROOT)

## 3. Подключение к Сфере

Возьми сервер, порт, базу и логин из свойств рабочего подключения DBeaver. Пароль вводится скрыто и не записывается в notebook.

In [ ]:
SPHERE_HOST = ''       # сервер из DBeaver
SPHERE_PORT = 5432     # порт из DBeaver
SPHERE_DATABASE = ''   # база данных из DBeaver
SPHERE_USER = ''       # логин из DBeaver

if not all([SPHERE_HOST, SPHERE_DATABASE, SPHERE_USER]):
    raise ValueError('Заполни SPHERE_HOST, SPHERE_DATABASE и SPHERE_USER')

password = getpass.getpass('Пароль от Сферы: ')
connection_url = URL.create(
    drivername='postgresql+psycopg',
    username=SPHERE_USER,
    password=password,
    host=SPHERE_HOST,
    port=SPHERE_PORT,
    database=SPHERE_DATABASE,
)
engine = create_engine(connection_url, pool_pre_ping=True)

In [ ]:
with engine.connect() as connection:
    connection_check = pd.read_sql_query(
        text('select current_database() as database_name, current_user as user_name'),
        connection,
    )

display(connection_check)

## 4. Подключение к Oracle КХД

Параметры возьми из рабочего Oracle-подключения DBeaver. Это второе независимое подключение внутри того же notebook. Пароль КХД вводится скрыто и не сохраняется.

In [ ]:
KHD_HOST = ''          # сервер Oracle из DBeaver
KHD_PORT = 1521        # порт Oracle из DBeaver
KHD_SERVICE_NAME = ''  # Service name из DBeaver
KHD_USER = ''          # логин Oracle из DBeaver
KHD_DATA_SCHEMA = 'DM_RISK_AVATAR'

if not all([KHD_HOST, KHD_SERVICE_NAME, KHD_USER]):
    raise ValueError('Заполни KHD_HOST, KHD_SERVICE_NAME и KHD_USER')

khd_password = getpass.getpass('Пароль от КХД: ')
khd_dsn = oracledb.makedsn(
    KHD_HOST,
    KHD_PORT,
    service_name=KHD_SERVICE_NAME,
)
khd_connection = oracledb.connect(
    user=KHD_USER,
    password=khd_password,
    dsn=khd_dsn,
)

In [ ]:
with khd_connection.cursor() as cursor:
    cursor.execute(
        "select user, sys_context('USERENV', 'DB_NAME') from dual"
    )
    khd_user_name, khd_database_name = cursor.fetchone()

    cursor.execute(
        """
        select table_name
        from all_tables
        where owner = :owner
          and table_name in ('BUILDINGS', 'EGRN_DATA')
        order by table_name
        """,
        owner=KHD_DATA_SCHEMA.upper(),
    )
    available_khd_tables = [row[0] for row in cursor.fetchall()]

print('Пользователь КХД:', khd_user_name)
print('База КХД:', khd_database_name)
print('Доступные таблицы:', available_khd_tables)

expected_khd_tables = {'BUILDINGS', 'EGRN_DATA'}
missing_khd_tables = sorted(expected_khd_tables - set(available_khd_tables))
if missing_khd_tables:
    raise PermissionError(
        'Не видны таблицы КХД: ' + ', '.join(missing_khd_tables)
    )

## 5. SQL 41 и его запуск

Запрос может выполняться дольше тестового запроса подключения. Не закрывай kernel во время выполнения.

In [ ]:
sql = "/*\nДля чего нужен запрос\n---------------------\nЗапрос собирает основу датасета для модели 1 по недвижимости ЮЛ.\nОн объединяет сведения о договоре, объекте, адресе, страхователе,\nотрасли, страховых суммах и ближайшем предыдущем договоре.\n\nОдна строка результата\n----------------------\nОдна строка - один объект недвижимости в одном договоре.\nОдин договор может занимать несколько строк, если в нем несколько объектов.\n\n\nКак связаны таблицы\n-------------------\nДоговор -> заявка -> задача оформления -> объект в задаче\n        -> характеристики объекта -> сам объект -> адрес\n        -> условия страхования объекта\n\nИз договора берется страхователь. Из заявки берется CRM-карточка,\nв которой находятся отрасль и сегмент.\n\nКакие записи попадают в результат\n---------------------------------\n- задача оформления договора: task_type = draft_contract;\n- завершенная рабочая задача: status = operational_archive;\n- тип документа: ins_document_type = new_ins_contract,\n  ins_contract_prolong или NULL, то есть новый договор, пролонгация\n  или незаполненное значение;\n- отказ в страховании не установлен: ins_refuse IS NOT TRUE;\n- дата удаления отсутствует: d_delete IS NULL для задачи, заявки,\n  договора и объекта;\n- тип объекта: elementary_obj_type = nedv_ul_and_ip,\n  то есть недвижимость ЮЛ и ИП.\n\n\nКак читать страховые суммы\n--------------------------\n- contract_insured_sum - общая СС всего договора;\n- task_object_insured_sum - СС объекта в строке связи задачи и объекта;\n- condition_min_insured_sum и condition_max_insured_sum - минимальная и\n  максимальная СС среди условий выбранной версии объекта;\n- insured_sum - СС среди условий выбранной версии объекта.\n\n\nКак используется история\n-------------------------\nБлижайший предыдущий договор ищется по bps_contract.prevcontract_id.\nПрошлая СС объекта заполняется только тогда, когда в текущем и предыдущем\nдоговоре совпал object_id. Если при пролонгации объект завели с новым ID,\nпрошлая объектная СС останется пустой.\n\n*/\n\nwith task_candidates as (\n    /* Шаг 1. Находим все подходящие задачи оформления. */\n    select\n        c.id as contract_id,\n        c.n_contract as contract_number,\n        c.prevcontract_id as previous_contract_id,\n        c.rootcontract_id as root_contract_id,\n        c.contractor_id as policyholder_id,\n        c.document_status as contract_status,\n        c.d_sign_contract as contract_sign_date,\n        c.d_start_contract as contract_start_date,\n        c.d_end_contract as contract_end_date,\n        c.currency as contract_currency,\n        c.ins_product_sbs as insurance_product,\n        c.ins_program as insurance_program,\n\n        r.id as request_id,\n        r.corporate_crm_id,\n        r.business_segment,\n\n        t.id as task_id,\n        t.d_create as task_create_date,\n        t.d_change as task_change_date,\n        t.task_type,\n        t.status as task_status,\n        t.ins_document_type,\n        t.contract_type,\n        t.d_conclusion_ins_contract as contract_conclusion_date,\n        t.ins_refuse,\n        t.industry as task_industry,\n        t.subindustry as task_subindustry,\n        t.total_ins_contract_amount as contract_insured_sum,\n        t.total_ins_contract_premium as contract_premium,\n        t.curr_ins_contract_amount as contract_amount_currency,\n\n        coalesce(\n            t.d_conclusion_ins_contract::timestamp with time zone,\n            c.d_sign_contract,\n            t.d_create\n        ) as as_of_date,\n\n        row_number() over (\n            partition by c.id\n            order by\n                coalesce(\n                    t.d_conclusion_ins_contract::timestamp with time zone,\n                    t.d_create,\n                    t.d_change\n                ) desc nulls last,\n                t.d_create desc nulls last,\n                t.d_change desc nulls last,\n                t.id desc\n        ) as task_number\n    from bps_request_ins_task t\n    join bps_request_ins r\n        on r.id = t.request_ins_id\n    join bps_contract c\n        on c.id = r.contract_id\n    where t.task_type = 'draft_contract'\n      and t.status = 'operational_archive'\n      and (\n          t.ins_document_type = 'new_ins_contract'\n          or t.ins_document_type = 'ins_contract_prolong'\n          or t.ins_document_type is null\n      )\n      and t.ins_refuse is not true\n      and t.d_delete is null\n      and r.d_delete is null\n      and c.d_delete is null\n),\n\nselected_tasks as (\n    /* Шаг 2. Для каждого договора оставляем одну самую позднюю задачу. */\n    select *\n    from task_candidates\n    where task_number = 1\n),\n\ncontract_context as (\n    /* Шаг 3. Добавляем ближайший предыдущий договор, если он указан. */\n    select\n        current_task.*,\n        previous_contract.n_contract as previous_contract_number,\n        previous_contract.d_start_contract as previous_contract_start_date,\n        previous_contract.d_end_contract as previous_contract_end_date,\n        previous_task.contract_insured_sum as previous_contract_insured_sum,\n        previous_task.contract_premium as previous_contract_premium,\n        previous_task.contract_amount_currency\n            as previous_contract_amount_currency\n    from selected_tasks current_task\n    left join bps_contract previous_contract\n        on previous_contract.id = current_task.previous_contract_id\n    left join selected_tasks previous_task\n        on previous_task.contract_id = current_task.previous_contract_id\n),\n\nobject_candidates as (\n    /*\n    Шаг 4. К выбранной задаче присоединяем объекты недвижимости.\n    Здесь же добавляем адрес, страхователя, CRM и характеристики объекта.\n    */\n    select\n        contract.*,\n\n        policyholder.inn as policyholder_inn,\n        policyholder.contractor_type as policyholder_type,\n        policyholder.cdi_id as policyholder_cdi_id,\n        policyholder.ogrn as policyholder_ogrn,\n        policyholder.kpp as policyholder_kpp,\n        policyholder.company_name_short as policyholder_name,\n        policyholder.company_form as policyholder_company_form,\n        policyholder.company_register_day\n            as policyholder_registration_date,\n\n        crm.id as crm_id,\n        crm.client_id as crm_client_id,\n        crm.segment as crm_segment,\n        crm.macroindustry as crm_macroindustry,\n        crm.industry as crm_industry,\n        crm.primary_occupation as crm_primary_occupation,\n        crm.specialization as crm_specialization,\n        crm.okved as crm_okved,\n\n        link.id as task_object_link_id,\n        link.characteristics_id,\n        link.object_group_id,\n        link.insured_sum as task_object_insured_sum,\n        link.insured_sum_currency as task_object_insured_sum_currency,\n        link.per_occurance_limit as task_object_per_occurrence_limit,\n\n        obj.id as object_id,\n        obj.obj_name as object_name,\n        obj.description as object_description,\n        obj.obj_type as object_type,\n        obj.elementary_obj_type,\n        obj.original_address,\n        obj.geo_address_id,\n\n        ch.version_number as characteristics_version_number,\n        ch.version_start_date as characteristics_version_start_date,\n        ch.version_end_date as characteristics_version_end_date,\n        ch.version_is_active as characteristics_version_is_active,\n        ch.insurance_value,\n        ch.insurance_value_currency,\n        ch.insurance_value_basis,\n        ch.is_pledged,\n        ch.pledged_value,\n        ch.ownership_type,\n        ch.is_leased,\n        ch.insured_components,\n        ch.activity_types,\n        ch.risk_natures,\n        ch.insurance_territory,\n        ch.characteristics ->> 'total_area_sq_m' as total_area,\n        ch.characteristics ->> 'occupied_area_sq_m' as occupied_area,\n        ch.characteristics ->> 'construction_year' as construction_year,\n        ch.characteristics ->> 'last_capital_repair_year'\n            as capital_repair_year,\n        ch.characteristics ->> 'total_floors_count' as floors_count,\n        ch.characteristics ->> 'occupied_floor' as occupied_floor,\n        ch.characteristics ->> 'load_bearing_walls_material'\n            as walls_material,\n        ch.characteristics ->> 'interfloor_overlap_material'\n            as overlap_material,\n        ch.characteristics ->> 'roofing_material' as roofing_material,\n        ch.characteristics as object_characteristics_json,\n\n        address.full_address,\n        address.postal_code,\n        address.region_id as address_region_id,\n        address.area as district,\n        address.settlement,\n        address.street,\n        address.house,\n        address.building,\n        address.block,\n        address.flat,\n        address.office,\n        address.fias_code,\n        address.longitude,\n        address.latitude,\n        address.address_dgis_id,\n\n        row_number() over (\n            partition by contract.task_id, obj.id\n            order by\n                link.d_change desc nulls last,\n                link.d_create desc nulls last,\n                ch.version_start_date desc nulls last,\n                ch.version_number desc nulls last,\n                link.id desc,\n                ch.id desc\n        ) as object_number\n    from contract_context contract\n    join bps_request_ins_task_insurance_object link\n        on link.parent_id = contract.task_id\n    join base_insurance_object_characteristics ch\n        on ch.id = link.characteristics_id\n    join base_insurance_object obj\n        on obj.id = ch.insurance_object_id\n    left join base_geo_address address\n        on address.id = obj.geo_address_id\n    left join bps_contractor policyholder\n        on policyholder.id = contract.policyholder_id\n    left join bps_corporate_crm crm\n        on crm.id = contract.corporate_crm_id\n    where obj.elementary_obj_type = 'nedv_ul_and_ip'\n      and obj.d_delete is null\n),\n\nselected_objects as (\n    /*\n    Шаг 5. Если объект несколько раз связан с одной задачей,\n    оставляем одну самую позднюю запись связи.\n    */\n    select *\n    from object_candidates\n    where object_number = 1\n),\n\nselected_characteristics as (\n    /* Шаг 6. Получаем список версий объектов для поиска их условий. */\n    select distinct characteristics_id\n    from selected_objects\n),\n\ncondition_summary as (\n    /*\n    Шаг 7. У одной версии объекта может быть несколько вариантов условий.\n    Сворачиваем их в одну строку, чтобы один объект не продублировался.\n    */\n    select\n        cond.characteristics_id,\n        count(*) as condition_count,\n        min(cond.insured_sum) as condition_min_insured_sum,\n        max(cond.insured_sum) as condition_max_insured_sum,\n        count(distinct cond.insured_sum_currency) filter (\n            where cond.insured_sum_currency is not null\n        ) as condition_currency_count,\n        string_agg(\n            distinct cond.insured_sum_currency,\n            ', '\n            order by cond.insured_sum_currency\n        ) filter (\n            where cond.insured_sum_currency is not null\n        ) as insured_sum_currency,\n        min(cond.per_occurance_limit) as minimum_per_occurrence_limit,\n        max(cond.per_occurance_limit) as maximum_per_occurrence_limit,\n        jsonb_agg(\n            jsonb_strip_nulls(\n                jsonb_build_object(\n                    'option_number', cond.terms_option_number,\n                    'insured_sum', cond.insured_sum,\n                    'currency', cond.insured_sum_currency,\n                    'per_occurrence_limit', cond.per_occurance_limit\n                )\n            )\n            order by cond.terms_option_number nulls last, cond.id\n        ) as conditions_json\n    from base_insurance_object_conditions cond\n    join selected_characteristics selected\n        on selected.characteristics_id = cond.characteristics_id\n    group by cond.characteristics_id\n),\n\nobject_data as (\n    /* Шаг 8. Добавляем к каждому объекту найденные суммы и условия. */\n    select\n        obj.*,\n        conditions.condition_count,\n        conditions.condition_min_insured_sum,\n        conditions.condition_max_insured_sum,\n        conditions.condition_currency_count,\n        conditions.insured_sum_currency,\n        conditions.minimum_per_occurrence_limit,\n        conditions.maximum_per_occurrence_limit,\n        conditions.conditions_json\n    from selected_objects obj\n    left join condition_summary conditions\n        on conditions.characteristics_id = obj.characteristics_id\n),\n\nobjects_with_previous as (\n    /*\n    Шаг 9. Ищем тот же object_id в ближайшем предыдущем договоре\n    и, если нашли, добавляем его предыдущую СС.\n    */\n    select\n        current_object.*,\n        case\n            when previous_object.condition_min_insured_sum =\n                 previous_object.condition_max_insured_sum\n             and previous_object.condition_currency_count <= 1\n            then previous_object.condition_max_insured_sum\n        end as previous_object_insured_sum,\n        previous_object.insured_sum_currency\n            as previous_object_insured_sum_currency\n    from object_data current_object\n    left join object_data previous_object\n        on previous_object.contract_id = current_object.previous_contract_id\n       and previous_object.object_id = current_object.object_id\n)\n\n/* Шаг 10. Формируем итоговый набор колонок. */\nselect\n    /* Основные ID. */\n    obj.contract_id,\n    obj.contract_number,\n    obj.previous_contract_id,\n    obj.root_contract_id,\n    obj.request_id,\n    obj.task_id,\n    obj.task_object_link_id,\n    obj.characteristics_id,\n    obj.object_id,\n    obj.geo_address_id,\n    obj.policyholder_id,\n    obj.corporate_crm_id,\n\n    /* Договор и его даты. */\n    obj.as_of_date,\n    obj.contract_conclusion_date,\n    obj.contract_sign_date,\n    obj.contract_start_date,\n    obj.contract_end_date,\n    obj.contract_status,\n    obj.ins_document_type,\n    obj.contract_type,\n    obj.contract_currency,\n    obj.insurance_product,\n    obj.insurance_program,\n\n    /* Объект. */\n    count(*) over (\n        partition by obj.contract_id\n    ) as real_estate_objects_in_contract,\n    obj.object_group_id,\n    obj.object_name,\n    obj.object_description,\n    obj.object_type,\n    obj.elementary_obj_type,\n    obj.total_area,\n    obj.occupied_area,\n    obj.construction_year,\n    obj.capital_repair_year,\n    obj.floors_count,\n    obj.occupied_floor,\n    obj.walls_material,\n    obj.overlap_material,\n    obj.roofing_material,\n    obj.ownership_type,\n    obj.is_leased,\n    obj.insured_components,\n    obj.activity_types,\n    obj.risk_natures,\n    obj.insurance_territory,\n\n    /* Адрес. */\n    obj.full_address,\n    obj.original_address,\n    obj.postal_code,\n    obj.address_region_id,\n    obj.district,\n    obj.settlement,\n    obj.street,\n    obj.house,\n    obj.building,\n    obj.block,\n    obj.flat,\n    obj.office,\n    obj.fias_code,\n    obj.longitude,\n    obj.latitude,\n    obj.address_dgis_id,\n\n    /* Страхователь, отрасль и сегмент. */\n    obj.policyholder_inn,\n    obj.policyholder_type,\n    obj.policyholder_cdi_id,\n    obj.policyholder_ogrn,\n    obj.policyholder_kpp,\n    obj.policyholder_name,\n    obj.policyholder_company_form,\n    obj.policyholder_registration_date,\n    obj.crm_id,\n    obj.crm_client_id,\n    (obj.crm_client_id = obj.policyholder_id) as crm_client_is_policyholder,\n    obj.crm_segment,\n    obj.crm_macroindustry,\n    obj.crm_industry,\n    obj.crm_primary_occupation,\n    obj.crm_specialization,\n    obj.crm_okved,\n    obj.business_segment,\n    obj.task_industry,\n    obj.task_subindustry,\n\n    /*\n    Все СС стоят рядом.\n    СС договора относится ко всему договору и повторяется у его объектов.\n    */\n    obj.contract_insured_sum,\n    obj.contract_amount_currency,\n    min(obj.condition_min_insured_sum) over (\n        partition by obj.contract_id\n    ) as contract_real_estate_min_insured_sum,\n    max(obj.condition_max_insured_sum) over (\n        partition by obj.contract_id\n    ) as contract_real_estate_max_insured_sum,\n    obj.task_object_insured_sum,\n    obj.task_object_insured_sum_currency,\n    obj.condition_min_insured_sum,\n    obj.condition_max_insured_sum,\n    case\n        /* Не выбираем случайную СС, если в условиях есть расхождения. */\n        when obj.condition_min_insured_sum =\n             obj.condition_max_insured_sum\n         and obj.condition_currency_count <= 1\n        then obj.condition_max_insured_sum\n    end as insured_sum,\n    obj.insured_sum_currency,\n    obj.condition_currency_count,\n    obj.previous_contract_insured_sum,\n    obj.previous_contract_amount_currency,\n    obj.previous_object_insured_sum,\n    obj.previous_object_insured_sum_currency,\n\n    /* Премии, стоимости и лимиты. */\n    obj.contract_premium,\n    obj.previous_contract_premium,\n    obj.insurance_value,\n    obj.insurance_value_currency,\n    obj.insurance_value_basis,\n    obj.is_pledged,\n    obj.pledged_value,\n    obj.task_object_per_occurrence_limit,\n    obj.minimum_per_occurrence_limit,\n    obj.maximum_per_occurrence_limit,\n\n    /* Простая договорная история. */\n    (obj.previous_contract_id is not null) as has_previous_contract,\n    obj.previous_contract_number,\n    obj.previous_contract_start_date,\n    obj.previous_contract_end_date,\n\n    /* Исходные данные для проверки. */\n    obj.condition_count,\n    obj.conditions_json,\n    obj.characteristics_version_number,\n    obj.characteristics_version_start_date,\n    obj.characteristics_version_end_date,\n    obj.characteristics_version_is_active,\n    obj.object_characteristics_json,\n    obj.task_type,\n    obj.task_status,\n    obj.ins_refuse\nfrom objects_with_previous obj\norder by\n    obj.as_of_date desc nulls last,\n    obj.contract_id,\n    obj.object_id;\n"

with engine.connect() as connection:
    strict_df = pd.read_sql_query(text(sql), connection)

print('Строк:', len(strict_df))
print('Колонок:', len(strict_df.columns))
display(strict_df.head(3))

## 6. Проверка зерна и заполненности

Здесь выводятся только количества. Адреса, ИНН и номера договоров не печатаются.

In [ ]:
required_columns = {
    'contract_id', 'task_id', 'object_id', 'characteristics_id',
    'elementary_obj_type', 'insured_sum', 'full_address'
}
missing_columns = sorted(required_columns - set(strict_df.columns))
if missing_columns:
    raise ValueError('Не найдены ожидаемые колонки: ' + ', '.join(missing_columns))

profile = pd.DataFrame({
    'Показатель': [
        'Строк',
        'Уникальных договоров',
        'Уникальных задач',
        'Уникальных объектов',
        'Уникальных пар задача + объект',
        'Строк с target',
        'Строк с адресом',
        'Строк с пустым типом объекта',
    ],
    'Значение': [
        len(strict_df),
        strict_df['contract_id'].nunique(dropna=True),
        strict_df['task_id'].nunique(dropna=True),
        strict_df['object_id'].nunique(dropna=True),
        strict_df[['task_id', 'object_id']].drop_duplicates().shape[0],
        strict_df['insured_sum'].notna().sum(),
        strict_df['full_address'].fillna('').str.strip().ne('').sum(),
        strict_df['elementary_obj_type'].fillna('').str.strip().eq('').sum(),
    ],
})

display(profile)

In [ ]:
duplicate_keys = (
    strict_df.groupby(['task_id', 'object_id'], dropna=False)
    .size()
    .gt(1)
    .sum()
)
print('Повторных ключей задача + объект:', duplicate_keys)
display(strict_df['elementary_obj_type'].fillna('empty').value_counts(dropna=False))

## 7. Сохранение результата

CSV сохраняется в локальную папку, которая исключена из Git. Кодировка `utf-8-sig` и разделитель `;` подходят для русского Excel.

In [ ]:
output_path = OUTPUT_DIR / 'датасет_41_строгий.csv'
strict_df.to_csv(output_path, index=False, sep=';', encoding='utf-8-sig')
print('Сохранено:', output_path)

In [ ]:
engine.dispose()
khd_connection.close()
print('Подключения к Сфере и КХД закрыты')